# Run Compute Tasks

This notebook is not a tutorial or explainer.  It is simply a scaffold for running tasks.

The first few cells collect the details required for what to run and where to run it, and the final cell pulls it all together.

In [ ]:
# Imports; the usual Python first step
import time
from globus_compute_sdk import Executor

In [ ]:
# Define endpoint IDs with a useful variable name
globus_compute_tutorial_endpoint_id = "4b116d3c-1703-4f8f-9f6f-39921e5864df"
your_endpoint_id = "<put_your_ep_uuid_here>"

In [ ]:
# Pick the endpoint ID of interest:
endpoint_id = globus_compute_tutorial_endpoint_id

In [ ]:
# Define your module WITHIN this string -- the last cell uses .register_source_code() (not .register_function) 
python_module_text = """
import os
import typing as t

import psutil

def row_by_row(data: t.Iterable[tuple[t.Any, t.Any]]) -> str:
    lines = []
    for ndx, (key, val) in enumerate(data):
        c = ndx % 2 and "\033[92;1m" or "\033[93;1m"
        lines.append(f"{c}{key:>19}: {val}\033[0m")
    return "\\n".join(lines)


def peek_at_host() -> dict[str, t.Any]:
    return {
      "core_count": os.cpu_count(),
      "physical_core_count": psutil.cpu_count(logical=False),
      "cores_available": os.sched_getaffinity(0),
      "virtmem": psutil.virtual_memory(),
      "uname": os.uname(),
    }


def pretty_print_peek_at_host() -> str:
    return row_by_row(peek_at_host().items())
"""

In [ ]:
# Replace this string with the function to call within the module defined in the previous cell
module_entrypoint = "pretty_print_peek_at_host"

module_desc = "[optional] describe the function's intention or other useful-to-you-later details"
module_desc = None  # Remove or comment this line if you want a description

In [ ]:
# If the function needs arguments, specify them here
f_args = ()
f_kwargs = {}

In [ ]:
assert f"def {module_entrypoint}" in python_module_text, "Entrypoint not found!"

with Executor(endpoint_id) as ex:
    func_id = ex.register_source_code(python_module_text, module_entrypoint, description=module_desc)
    print(f"Module registered; entrypoint: `{module_entrypoint}` (function id: {func_id})")
    future = ex.submit_to_registered_function(func_id, args=f_args, kwargs=f_kwargs)
    print("Task enqueued, awaiting submission.")
    try:
        while not future.task_id:
            time.sleep(0.1)
        print(f"Task submitted ({future.task_id}); awaiting result:")
        print(future.result())
    except Exception as e:
        print("  \033[91;1mOh no!  Task failed.  The reported exception was\033[0m:")
        print(str(e))